# Supply Chain Service Level Tracking Analysis

In [1]:
# Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Cleaning

In [2]:
# Load data
customers = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_customers.csv")
date = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_date.csv")
products = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_products.csv")
targets = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_targets_orders.csv")
order_lines = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/fact_order_lines.csv")


### Preview data set

In [3]:
customers.head(3)

,customer_id,customer_name,city
0,789201,Rel Fresh,Surat
1,789202,Rel Fresh,Ahmedabad
2,789203,Rel Fresh,Vadodara


In [4]:
date.head(3)

,date,mmm_yy,week_no
0,01-Apr-22,01-Apr-22,W 14
1,03-Apr-22,01-Apr-22,W 15
2,04-Apr-22,01-Apr-22,W 15


In [5]:
products.head(3)

,product_id,product_name,category
0,25891101,AM Milk 500,Dairy
1,25891102,AM Milk 250,Dairy
2,25891103,AM Milk 100,Dairy


In [6]:
targets.head(3)

,customer_id,ontime_target%,infull_target%,otif_target%
0,789201,87,81,70
1,789202,85,81,69
2,789203,92,76,70


In [7]:
order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,"Tuesday, March 1, 2022",789203,25891601,110,"Friday, March 4, 2022","Friday, March 4, 2022",110
1,FMR32320302,"Tuesday, March 1, 2022",789320,25891203,347,"Wednesday, March 2, 2022","Wednesday, March 2, 2022",347
2,FMR33320501,"Tuesday, March 1, 2022",789320,25891203,187,"Thursday, March 3, 2022","Thursday, March 3, 2022",150


### Standardise column names for all tables

In [8]:
# Standardise column names
for df in [customers, date, products, targets, order_lines]:
    df.columns = df.columns.str.lower().str.strip()

### Validate **`customers`** table
- data types
- missing values
- duplicates

In [9]:
customers.dtypes

customer_id       int64
customer_name    object
city             object
dtype: object

In [10]:
customers.isnull().sum()

customer_id      0
customer_name    0
city             0
dtype: int64

In [11]:
customers.duplicated().sum()

np.int64(0)

In [12]:
customers.duplicated(subset=['customer_id', 'customer_name', 'city']).sum()

np.int64(0)

### Validate **`date`** table
- data types
- missing values
- duplicates

In [13]:
date.dtypes

date       object
mmm_yy     object
week_no    object
dtype: object

#### parse datetime data type for columns:
- date
- mmm-yy

In [14]:
# Convert date column
date['date'] = pd.to_datetime(date['date'], format='%d-%b-%y')

# Convert mmm_yy to datetime (month-level)
date['mmm_yy'] = pd.to_datetime(date['mmm_yy'], format='%d-%b-%y')

# week_no stays as text
date['week_no'] = date['week_no'].astype('string')

date.dtypes

date       datetime64[ns]
mmm_yy     datetime64[ns]
week_no    string[python]
dtype: object

In [15]:
date.head(3)

,date,mmm_yy,week_no
0,2022-04-01,2022-04-01,W 14
1,2022-04-03,2022-04-01,W 15
2,2022-04-04,2022-04-01,W 15


In [16]:
date.isnull().sum()

date       0
mmm_yy     0
week_no    0
dtype: int64

In [17]:
date.duplicated().sum()

np.int64(0)

### Validate **`products`** table
- data types
- missing values
- duplicates

In [18]:
products.dtypes

product_id       int64
product_name    object
category        object
dtype: object

In [19]:
products.isnull().sum()

product_id      0
product_name    0
category        0
dtype: int64

In [20]:
products.duplicated(subset=['product_id', 'product_name']).sum()

np.int64(0)

### Validate **`targets`** table
- data types
- missing values
- duplicates

In [21]:
targets.dtypes

customer_id       int64
ontime_target%    int64
infull_target%    int64
otif_target%      int64
dtype: object

In [22]:
targets.isnull().sum()

customer_id       0
ontime_target%    0
infull_target%    0
otif_target%      0
dtype: int64

In [23]:
targets.duplicated(subset=['customer_id']).sum()

np.int64(0)

In [24]:
# Adjust column names
targets = targets.rename(columns={
    'ontime_target%': 'ontime_target_pct',
    'infull_target%': 'infull_target_pct',
    'otif_target%': 'otif_target_pct'
})

In [25]:
targets.columns.to_list()

['customer_id', 'ontime_target_pct', 'infull_target_pct', 'otif_target_pct']

### Validate **`order_lines`** table
- data types
- missing values
- duplicates

In [26]:
order_lines.dtypes

order_id                object
order_placement_date    object
customer_id              int64
product_id               int64
order_qty                int64
agreed_delivery_date    object
actual_delivery_date    object
delivery_qty             int64
dtype: object

#### parse datetime data type for columns:
- order_placement_date
- agreed_delivery_date
- actual_delivery_date

In [27]:
order_lines['order_placement_date'] = pd.to_datetime(order_lines['order_placement_date'])
order_lines['agreed_delivery_date'] = pd.to_datetime(order_lines['agreed_delivery_date'])
order_lines['actual_delivery_date'] = pd.to_datetime(order_lines['actual_delivery_date'])

order_lines.dtypes

order_id                        object
order_placement_date    datetime64[ns]
customer_id                      int64
product_id                       int64
order_qty                        int64
agreed_delivery_date    datetime64[ns]
actual_delivery_date    datetime64[ns]
delivery_qty                     int64
dtype: object

In [28]:
order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150


In [29]:
order_lines.isnull().sum()

order_id                0
order_placement_date    0
customer_id             0
product_id              0
order_qty               0
agreed_delivery_date    0
actual_delivery_date    0
delivery_qty            0
dtype: int64

In [30]:
order_lines.duplicated().sum()

np.int64(0)

### Merge  **`orders_lines`** with `customers` & `products`

In [31]:
order_lines = (order_lines.merge(customers, on='customer_id', how='left').merge(products, on='product_id', how='left'))

order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy


In [32]:
order_lines.shape

(57096, 12)

### Create Core Service Level Agreement (SLA) Flags
- On-Time Delivery
- In-Full Delivery
- OTIF (On-Time + In-Full)

This makes it possible to compute:
- On-Time vs Target
- In-Full vs Target
- OTIF vs Target

In [33]:
# On-Time Delivery
order_lines['on_time_flag'] = (
    order_lines['actual_delivery_date'] 
    <= order_lines['agreed_delivery_date']
)

# In-Full Delivery
order_lines['in_full_flag'] = (
    order_lines['delivery_qty'] 
    >= order_lines['order_qty']
)

# OTIF (On-Time + In-Full)
order_lines['otif_flag'] = (
    order_lines['on_time_flag'] & order_lines['in_full_flag']
)

order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category,on_time_flag,in_full_flag,otif_flag
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages,True,True,True
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy,True,True,True
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy,True,False,False


### Fill Rate Metrics

#### 1. Line Fill Rate (LiFR)

**Definition**: Number of order lines fulfilled in full / total order lines

So at line level, we first create a **`line_fulfilled_flag`**:

**LiFR** is calculated later during aggregation as: **`SUM(line_fulfilled_flag)` / `COUNT(order_lines)`**

In [34]:
# line_fulfilled_flag - boolean (true/false)
order_lines['line_fulfilled_flag'] = (order_lines['delivery_qty'] >= order_lines['order_qty'])

# Convert to numeric (needed for aggregation later)
order_lines['line_fulfilled_flag'] = (order_lines['line_fulfilled_flag'].astype(int))


#### 2. Volume Fill Rate (VoFR)

At line level:

**Definition**: Total quantity delivered / total quantity ordered

Final **VoFR** is calculated later as: **`SUM(delivery_qty)` / `SUM(order_qty)`**


In [35]:
# VoFR (line level)
order_lines['volume_fill_rate'] = (order_lines['delivery_qty'] / order_lines['order_qty']).round(2)

### Delay Calculation
- `delay_days`
- `late_delivery_flag`

In [36]:
# delay_days
order_lines['delay_days'] = (order_lines['actual_delivery_date'] - order_lines['agreed_delivery_date']).dt.days

# late_delivery_flag
order_lines['late_delivery_flag'] = order_lines['delay_days'] > 0

### Date Enrichment:

Extract time features for drilling

In [37]:
order_lines['year'] = order_lines['order_placement_date'].dt.year
order_lines['month'] = order_lines['order_placement_date'].dt.month
order_lines['week'] = order_lines['order_placement_date'].dt.isocalendar().week
order_lines['day'] = order_lines['order_placement_date'].dt.date

In [38]:
order_lines.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,...,in_full_flag,otif_flag,line_fulfilled_flag,volume_fill_rate,delay_days,late_delivery_flag,year,month,week,day
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,...,True,True,1,1.0,0,False,2022,3,9,2022-03-01
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,...,True,True,1,1.0,0,False,2022,3,9,2022-03-01
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,...,False,False,0,0.8,0,False,2022,3,9,2022-03-01


### Validate unique values per **`order_id`** for selected fields;
- customer_id
- customer_name
- city
- order_placement_date
- year
- month

In [39]:
# Count unique values per order_id for each column
validation_counts = (
    order_lines
    .groupby('order_id')
    .agg({
        'customer_id': 'nunique',
        'customer_name': 'nunique',
        'city': 'nunique',
        'order_placement_date': 'nunique',
        'year': 'nunique',
        'month': 'nunique'
    })
)

# Show the maximum unique count per column
validation_summary = validation_counts.max()

validation_summary

customer_id             1
customer_name           1
city                    1
order_placement_date    1
year                    1
month                   1
dtype: int64

### Aggregate **`order_lines`** in a new DataFrame **`orders_aggregate`**

In [40]:
orders_aggregate = order_lines.groupby('order_id').agg({
    'customer_id': 'first',
    'customer_name': 'first',
    'city': 'first',
    'order_placement_date': 'first',
    'year': 'first',
    'month': 'first',
    'on_time_flag': 'all',
    'in_full_flag': 'all'
}).reset_index()

# derive OTIF; OTIF = On-Time AND In-Full
orders_aggregate['otif_flag'] = orders_aggregate['on_time_flag'] & orders_aggregate['in_full_flag']

In [41]:
orders_aggregate.head(3)

,order_id,customer_id,customer_name,city,order_placement_date,year,month,on_time_flag,in_full_flag,otif_flag
0,FAP410101302,789101,Vijay Stores,Surat,2022-04-08,2022,4,True,False,False
1,FAP410101402,789101,Vijay Stores,Surat,2022-04-07,2022,4,True,False,False
2,FAP410101502,789101,Vijay Stores,Surat,2022-04-09,2022,4,True,True,True


In [42]:
orders_aggregate.shape

(31729, 10)

## Data Exploration

### Total Orders

In [50]:
total_orders = orders_aggregate['order_id'].nunique()
total_orders

31729

### Total Order Lines

In [51]:
total_order_lines = order_lines.shape[0]
total_order_lines

57096

### Core Operational KPIs

In [44]:
kpi_summary = {
    'On-Time Rate': f"{orders_aggregate['on_time_flag'].mean() * 100:.1f}%",
    'In-Full Rate': f"{orders_aggregate['in_full_flag'].mean() * 100:.1f}%",
    'OTIF Rate': f"{orders_aggregate['otif_flag'].mean() * 100:.1f}%",
    'Line Fill Rate': f"{order_lines['line_fulfilled_flag'].mean() * 100:.1f}%",
    'Volume Fill Rate': f"{order_lines['volume_fill_rate'].mean() * 100:.1f}%"
}

kpi_summary

{'On-Time Rate': '59.0%',
 'In-Full Rate': '52.8%',
 'OTIF Rate': '29.0%',
 'Line Fill Rate': '66.0%',
 'Volume Fill Rate': '96.6%'}

### Mean Targets

In [45]:
sla_targets = {
    'On-Time Target': f"{targets['ontime_target_pct'].mean():.1f}%",
    'In-Full Target': f"{targets['infull_target_pct'].mean():.1f}%",
    'OTIF Target': f"{targets['otif_target_pct'].mean():.1f}%"
}

sla_targets

{'On-Time Target': '86.1%', 'In-Full Target': '76.5%', 'OTIF Target': '65.9%'}

### KPIs vs Mean Targets

In [46]:
gap_analysis = {
    'On-Time Gap': f"{(orders_aggregate['on_time_flag'].mean() - targets['ontime_target_pct'].mean()/100) * 100:.1f}%",
    'In-Full Gap': f"{(orders_aggregate['in_full_flag'].mean() - targets['infull_target_pct'].mean()/100) * 100:.1f}%",
    'OTIF Gap': f"{(orders_aggregate['otif_flag'].mean() - targets['otif_target_pct'].mean()/100) * 100:.1f}%"
}

gap_analysis

{'On-Time Gap': '-27.1%', 'In-Full Gap': '-23.7%', 'OTIF Gap': '-36.9%'}

Step 3 — City-Level Performance (Very High Value)

City is one of your strongest dimensions.

In [45]:
city_perf = (
    order_lines
    .groupby('city')
    .agg({
        'on_time_flag':'mean',
        'in_full_flag':'mean',
        'otif_flag':'mean',
        'delay_days':'mean',
        'order_id':'count'
    })
    .rename(columns={'order_id':'line_count'})
    .sort_values('otif_flag')
)

city_perf.head(10)

,on_time_flag,in_full_flag,otif_flag,delay_days,line_count
city,,,,,
Vadodara,0.699101,0.636939,0.452242,0.443355,19578
Ahmedabad,0.700041,0.675595,0.483279,0.446026,19676
Surat,0.736689,0.666853,0.505381,0.379105,17842


Step 4 — Customer Performance (Extremely Valuable)

This is business-critical.

In [46]:
customer_perf = (
    order_lines
    .groupby('customer_name')
    .agg({
        'on_time_flag':'mean',
        'in_full_flag':'mean',
        'otif_flag':'mean',
        'delay_days':'mean',
        'order_id':'count'
    })
    .rename(columns={'order_id':'line_count'})
    .sort_values('otif_flag')
)

customer_perf.head(10)

,on_time_flag,in_full_flag,otif_flag,delay_days,line_count
customer_name,,,,,
Coolblue,0.268125,0.515279,0.137507,1.270821,3338
Acclaimed Stores,0.268918,0.589327,0.152387,1.252241,4797
Lotus Mart,0.257290,0.600821,0.160575,1.283778,4870
Info Stores,0.831422,0.530524,0.434769,0.196777,3227
Elite Mart,0.847442,0.527406,0.448843,0.163216,3284
Sorefoz Mart,0.847912,0.533984,0.455959,0.160317,3281
Vijay Stores,0.847331,0.592346,0.496073,0.166566,4965
Logic Stores,0.833896,0.743936,0.619589,0.191894,3257
Expression Stores,0.831263,0.752802,0.623145,0.210845,3301


Step 5 — Product & Category Analysis (Root Cause Layer)

Now look at products.

In [47]:
product_perf = (
    order_lines
    .groupby('product_name')
    .agg({
        'on_time_flag':'mean',
        'in_full_flag':'mean',
        'otif_flag':'mean',
        'delay_days':'mean',
        'order_id':'count'
    })
    .rename(columns={'order_id':'line_count'})
    .sort_values('otif_flag')
)

product_perf.head(10)

,on_time_flag,in_full_flag,otif_flag,delay_days,line_count
product_name,,,,,
AM Butter 500,0.703851,0.651895,0.463936,0.435513,3272
AM Butter 250,0.713280,0.635200,0.467200,0.428160,3125
AM Ghee 100,0.700452,0.657521,0.468689,0.446740,3098
AM Ghee 250,0.695937,0.652500,0.469375,0.442188,3200
AM Tea 100,0.716337,0.653159,0.472240,0.407466,3134
AM Biscuits 250,0.717200,0.651601,0.474262,0.424984,3186
AM Curd 50,0.713837,0.655475,0.474741,0.417948,3187
AM Milk 250,0.707851,0.659055,0.475133,0.436972,3197
AM Tea 250,0.711104,0.651607,0.476615,0.417436,3143


Then:

In [48]:
category_perf = (
    order_lines
    .groupby('category')
    .agg({
        'on_time_flag':'mean',
        'in_full_flag':'mean',
        'otif_flag':'mean',
        'delay_days':'mean'
    })
    .sort_values('otif_flag')
)

category_perf

,on_time_flag,in_full_flag,otif_flag,delay_days
category,,,,
beverages,0.713984,0.655428,0.475531,0.416129
Dairy,0.708080,0.659466,0.478318,0.428995
Food,0.720725,0.664325,0.488416,0.413041


Step 6 — Time Trend Exploration (Operational Insight)

Use:

- year
- month
- week

Start with month:

In [49]:
monthly_perf = (
    order_lines
    .groupby(['year','month'])
    .agg({
        'on_time_flag':'mean',
        'in_full_flag':'mean',
        'otif_flag':'mean',
        'delay_days':'mean'
    })
    .reset_index()
)

In [50]:
monthly_perf

,year,month,on_time_flag,in_full_flag,otif_flag,delay_days
0,2022,3,0.714505,0.657919,0.479959,0.414967
1,2022,4,0.713891,0.658051,0.477664,0.421187
2,2022,5,0.705412,0.662874,0.476322,0.434604
3,2022,6,0.708915,0.656939,0.478965,0.430930
4,2022,7,0.718018,0.658854,0.486319,0.413526
5,2022,8,0.706005,0.663080,0.477879,0.430453


Step 7 — Delay Analysis (Very High Diagnostic Value)

This column is powerful:

delay_days distribution

In [51]:
order_lines['delay_days'].describe()

count    57096.000000
mean         0.424198
std          0.936105
min         -1.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          3.000000
Name: delay_days, dtype: float64

Then:

In [52]:
late_dist = (
    order_lines
    .groupby('late_delivery_flag')
    .size()
)

late_dist

late_delivery_flag
False    40605
True     16491
dtype: int64

Step 8 — Volume Fill Rate Exploration (Very Important)

You created: volume_fill_rate

Now analyse:

In [53]:
order_lines['volume_fill_rate'].describe()

count    57096.000000
mean         0.965890
std          0.057501
min          0.780000
25%          0.950000
50%          1.000000
75%          1.000000
max          1.000000
Name: volume_fill_rate, dtype: float64

And:

In [54]:
category_fill = (
    order_lines
    .groupby('category')['volume_fill_rate']
    .mean()
    .sort_values()
)

category_fill

category
Dairy        0.965775
beverages    0.965808
Food         0.966433
Name: volume_fill_rate, dtype: float64

### Delay Days

In [58]:
# chart

In [ ]:
# Average (exempt -ve and 0)

In [71]:
order_lines['delay_days'].mean()

np.float64(0.4241978422306291)